# 04 - Approach A: Expanded Linear (Ridge regression)

**Inertia, Momentum Timing** - Sprint 3, Approach A

*Keep Daniel-Moskowitz's framework; give it more features.*

- **Framing:** predict $\hat{E}_t[r^{UMD}_{t+1}]$, scale by conditional variance (same as DM).
- **Features:** DM originals + market context (market vol, market drawdown, UMD drawdown, lagged market returns).
- **Model:** RidgeCV with 5-fold CV. L2 regularization handles collinear macro features.
- **Sample:** UMD + market features only --- allows training from 1928, which turned out to matter. An earlier variant added macro features (VIX, term spread, credit spread) but required training to start in 1990, leaving only 10 years before OOS begins, and Ridge over-regularized the small sample.
- **OOS:** expanding window, refit annually starting 2000-01.
- **Weight:** same as DM baseline, clipped to [-1, +3], 20 bps one-way costs.

Target to beat: **DM OOS** from notebook 03.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.features import build_features, feature_sets, DM_FEATURES, MARKET_FEATURES
from src.backtest import expanding_window_oos, weights_from_predictions, apply_weights
from src.evaluation import perf_table, sharpe_bootstrap_ci, alpha_table, subsample_table
from src.data import get_factor_panel
from src.inertia_style import (
    apply_style, C, FULL_COL,
    save_fig, save_table, legend_below,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1960-01-01'

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 1. Load feature panel

UMD + market features only; no FRED. Full training history back to 1928.

In [2]:
df = build_features(include_macro=False)
feat = DM_FEATURES + MARKET_FEATURES
complete = df[feat + ['UMD_next','UMD']].dropna()

print(f'Panel: {df.shape}, {df.index.min().date()} -> {df.index.max().date()}')
print(f'Expanded features ({len(feat)}): {feat}')
print(f'Complete panel starts: {complete.index.min().date()}')

Panel: (1179, 14), 1927-12-31 -> 2026-02-28
Expanded features (8): ['bear', 'mom_var6', 'bear_x_var', 'mkt_vol6', 'mkt_dd', 'umd_dd', 'mkt_ret_1m', 'mkt_ret_12m']
Complete panel starts: 1927-12-31


## 2. Ridge fit function

Features are standardized inside a Pipeline so scaling is refit per training window. RidgeCV picks the L2 penalty via 5-fold CV from a tight log-spaced grid (LOO-CV is too noisy for small windows).

In [3]:
ALPHA_GRID = np.logspace(-2, 2, 13)

def fit_ridge(X, y):
    return Pipeline([
        ('scale', StandardScaler()),
        ('ridge', RidgeCV(alphas=ALPHA_GRID, cv=5)),
    ]).fit(X, y)

## 3. Run OOS backtest

In [4]:
preds = expanding_window_oos(
    complete, feat, 'UMD_next',
    fit_fn=fit_ridge, oos_start=OOS_START,
    refit_months=12, min_train_months=120,
)
w, c = weights_from_predictions(preds, df['UMD'], cap=(-1.0, 3.0))
print(f'OOS months: {len(preds)}   scaling c = {c:.2f}')
print(f'Prediction stats: mean={preds.mean():.4f}, std={preds.std():.4f}, range=[{preds.min():.3f},{preds.max():.3f}]')
print(f'Weight range: [{w.min():.2f}, {w.max():.2f}]   % short: {(w<0).mean()*100:.1f}%   % capped: {(w>=3).mean()*100:.1f}%')

back = apply_weights(w, df['UMD'], cost_bps_oneway=20.0)
back.head()

OOS months: 793   scaling c = 1.15
Prediction stats: mean=0.0085, std=0.0054, range=[-0.022,0.026]
Weight range: [-1.00, 3.00]   % short: 7.3%   % capped: 2.9%


,weight,r_next,r_gross,turnover,cost,r_net
date,,,,,,
1960-01-31,3.0000,0.0400,0.1200,3.0000,0.0060,0.1140
1960-02-29,1.2648,0.0141,0.0178,1.7352,0.0035,0.0144
1960-03-31,1.4744,0.0284,0.0419,0.2096,0.0004,0.0415
1960-04-30,1.4591,0.0479,0.0699,0.0153,0.0000,0.0699
1960-05-31,0.8608,0.0089,0.0077,0.5983,0.0012,0.0065


## 4. Rebuild the DM OOS benchmark for head-to-head

In [5]:
class OLSModel:
    def __init__(self, res): self.res = res
    def predict(self, X):
        return self.res.predict(sm.add_constant(X, has_constant='add'))

def fit_ols(X, y):
    return OLSModel(sm.OLS(y, sm.add_constant(X)).fit())

dm_preds = expanding_window_oos(df, DM_FEATURES, 'UMD_next', fit_fn=fit_ols,
                                 oos_start=OOS_START, refit_months=12, min_train_months=120)
dm_w, _ = weights_from_predictions(dm_preds, df['UMD'], cap=(-1.0, 3.0))
dm_back = apply_weights(dm_w, df['UMD'], cost_bps_oneway=20.0)

## 5. Performance comparison

In [6]:
idx = back.index.intersection(dm_back.index)
static_r = df['UMD_next'].reindex(idx)

returns = {
    'Static UMD':           static_r,
    'DM OOS (net)':         dm_back.loc[idx, 'r_net'],
    'Approach A (Ridge)':   back.loc[idx, 'r_net'],
}

perf = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(perf, '04_approach_a_performance', out_dir=TABLE_DIR)
perf

  (skipped tex for 04_approach_a_performance: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/04_approach_a_performance.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static UMD,793,0.0747,0.1422,0.5255,-1.3193,10.0409,-0.5782
DM OOS (net),793,0.1048,0.1419,0.7383,-0.3393,4.4136,-0.3394
Approach A (Ridge),793,0.0972,0.1404,0.6925,-0.1474,7.1891,-0.3167


In [7]:
boot = {name: sharpe_bootstrap_ci(r, n_boot=2000) for name, r in returns.items()}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '04_approach_a_sharpe_bootstrap', out_dir=TABLE_DIR)
boot_df

  (skipped tex for 04_approach_a_sharpe_bootstrap: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/04_approach_a_sharpe_bootstrap.{csv,md}


,sharpe,ci_low,ci_high
Static UMD,0.5255,0.2466,0.7944
DM OOS (net),0.7383,0.4667,0.9701
Approach A (Ridge),0.6925,0.4127,0.9208


## 6. Alphas vs FF5+UMD

In [8]:
factor_panel = get_factor_panel()
alpha_df = alpha_table(
    {k: v for k, v in returns.items() if k != 'Static UMD'},
    factor_panel, spec='FF5_UMD',
)
save_table(alpha_df, '04_approach_a_alpha_ff5umd', out_dir=TABLE_DIR)
alpha_df.T

  (skipped tex for 04_approach_a_alpha_ff5umd: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/04_approach_a_alpha_ff5umd.{csv,md}


,DM OOS (net),Approach A (Ridge)
alpha_monthly,0.0047,0.0051
alpha_annual,0.0560,0.0608
alpha_t,2.7307,2.8226
alpha_p,0.0063,0.0048
r2,0.0048,0.0090
n_obs,751.0000,751.0000
MKT_RF_beta,-0.0086,-0.0669
MKT_RF_t,-0.2414,-1.6384
SMB_beta,0.0531,-0.0127
SMB_t,0.9935,-0.2265


## 7. Subsample Sharpe

In [9]:
splits = {
    '1960-1979':    ('1960-01', '1979-12'),
    '1980-1999':    ('1980-01', '1999-12'),
    '2000-2009':    ('2000-01', '2009-12'),
    '2010-2019':    ('2010-01', '2019-12'),
    '2020-present': ('2020-01', None),
}
sub = subsample_table(returns, splits)
save_table(sub, '04_approach_a_subsample_sharpe', out_dir=TABLE_DIR)
sub

  (skipped tex for 04_approach_a_subsample_sharpe: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/04_approach_a_subsample_sharpe.{csv,md}


,1960-1979,1980-1999,2000-2009,2010-2019,2020-present
Static UMD,0.9604,0.9404,0.0106,0.3912,0.1302
DM OOS (net),1.0603,0.8307,0.4524,0.2523,0.5030
Approach A (Ridge),1.0379,0.8084,0.6616,0.0435,0.1769


## 8. Cumulative return

In [10]:
plot_df = pd.DataFrame(returns).dropna()
cum = (1 + plot_df).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.2))
ax.plot(cum.index, cum['Static UMD'],         color=C['muted'],  linewidth=1.0, label='Static UMD')
ax.plot(cum.index, cum['DM OOS (net)'],       color=C['purple'], linewidth=1.0, label='DM OOS')
ax.plot(cum.index, cum['Approach A (Ridge)'], color=C['blue'],   linewidth=1.3, label='Approach A (Ridge)')
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of $1 (log)')
ax.set_title('Approach A vs DM OOS vs Static UMD, 2000-present', loc='left', color=C['dark'])
legend_below(ax)
save_fig(fig, '09_approach_a_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/09_approach_a_cumret.{pdf,png}


## 9. Drawdown

In [11]:
def _dd(r):
    c = (1 + r).cumprod(); return c / c.cummax() - 1

fig, ax = plt.subplots(figsize=(FULL_COL, 2.8))
ax.plot(_dd(returns['Static UMD']).index,       _dd(returns['Static UMD']).values,       color=C['muted'],  linewidth=0.7, label='Static UMD')
ax.plot(_dd(returns['DM OOS (net)']).index,     _dd(returns['DM OOS (net)']).values,     color=C['purple'], linewidth=0.9, label='DM OOS')
dA = _dd(returns['Approach A (Ridge)'])
ax.fill_between(dA.index, dA.values, 0, color=C['blue'], alpha=0.18, linewidth=0)
ax.plot(dA.index, dA.values, color=C['blue'], linewidth=0.9, label='Approach A (Ridge)')
ax.axhline(0, color=C['dark'], linewidth=0.3)
ax.set_ylabel('Drawdown')
ax.set_title('Drawdowns: Approach A vs DM OOS vs Static UMD', loc='left', color=C['dark'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
legend_below(ax)
save_fig(fig, '10_approach_a_drawdown', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/10_approach_a_drawdown.{pdf,png}


## 10. Ridge coefficient inspection

Diagnostic fit on the full complete sample. Shows which features Ridge assigns weight to.

In [12]:
full_pipe = fit_ridge(complete[feat].values, complete['UMD_next'].values)
ridge = full_pipe.named_steps['ridge']

coef_df = pd.DataFrame({
    'feature': feat,
    'standardized_coef': ridge.coef_,
}).sort_values('standardized_coef', key=abs, ascending=False)
coef_df.loc[len(coef_df)] = {'feature': '(selected alpha)', 'standardized_coef': ridge.alpha_}
save_table(coef_df.set_index('feature'), '04_approach_a_ridge_coefficients', out_dir=TABLE_DIR)
coef_df

  (skipped tex for 04_approach_a_ridge_coefficients: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/04_approach_a_ridge_coefficients.{csv,md}


,feature,standardized_coef
6,mkt_ret_1m,-0.0054
4,mkt_dd,0.0045
7,mkt_ret_12m,0.0037
2,bear_x_var,-0.0021
1,mom_var6,0.0018
3,mkt_vol6,0.0006
0,bear,-0.0004
5,umd_dd,0.0003
8,(selected alpha),100.0000


## Takeaways --- Approach A

- Ridge with UMD + market features lands close to DM OOS but does not clearly beat it.
- Interpretation: Daniel-Moskowitz's three-variable OLS (`bear`, `mom_var6`, `bear_x_var`) already captures most of the timing signal in our feature set. Additional linear market-context features add noise as much as signal when shrunk by L2.
- This is an honest, defensible finding for the prospectus: *"the marginal value of more features, same functional form, is not statistically distinguishable from zero."*
- **Next:** Approach B reframes the problem as classification (predict P(crash), not E[r]); Approach C drops supervised labels entirely and uses an unsupervised HMM. Those are methodologically orthogonal --- they may succeed where Approach A did not.